# Train the Clarity Cancellation Model on Google Colab

Train on Colab's compute, then download the artifacts into your app's `backend/artifacts/` folder.

**You only need to upload 2 files from the repo** (Step 3):
- `backend/app/utils/feature_engineering.py`
- `backend/scripts/train_cancellation_model.py`

> **Version parity matters.** The model is pickled with specific library versions. Step 1 pins the **same** versions as `backend/requirements.txt` so the model loads cleanly in your app. If you change a version here, change it there too.

Recommended: **Runtime → Change runtime type → High-RAM** (the dataset is large).

## Step 1 — Install dependencies (pinned to match the app)

In [ ]:
!pip install -q \
    scikit-learn==1.5.2 \
    xgboost==2.1.3 \
    lightgbm==4.5.0 \
    optuna==4.1.0 \
    shap==0.46.0 \
    imbalanced-learn==0.12.4 \
    joblib==1.4.2 \
    pandas==2.2.3 \
    numpy==2.1.3 \
    db-dtypes==1.3.1 \
    tqdm==4.67.1 \
    google-cloud-bigquery==3.27.0 \
    google-cloud-bigquery-storage==2.27.0
print('Done. If Colab asks you to RESTART THE RUNTIME, do it, then re-run from Step 2.')

## Step 2 — Authenticate to Google Cloud / BigQuery
Use the same Google account that has access to the `long-ceiling-343505` project.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import os
# Project + dataset the training script reads vendor_kpi from
os.environ['GCP_PROJECT_ID'] = 'long-ceiling-343505'
os.environ['BQ_CALLS_DATASET'] = 'reports'
print('Authenticated. Project set to', os.environ['GCP_PROJECT_ID'])

## Step 3 — Upload the 2 source files
Run the cell, then pick **`feature_engineering.py`** and **`train_cancellation_model.py`** from your machine.
They must land in the same folder (`/content`).

In [ ]:
from google.colab import files
uploaded = files.upload()  # select feature_engineering.py AND train_cancellation_model.py
assert 'feature_engineering.py' in uploaded and 'train_cancellation_model.py' in uploaded, \
    'Please upload BOTH files.'
print('Uploaded:', list(uploaded.keys()))

## Step 4 — Train
Full run uses 50 Optuna trials. For a quick smoke test first, use `--limit 50000 --trials 10`.
Artifacts are written to `/content/artifacts/`.

In [ ]:
!python train_cancellation_model.py --trials 50
# Quick test instead:
# !python train_cancellation_model.py --limit 50000 --trials 10

## Step 5 — Download the trained artifacts
Zips everything in `artifacts/` and downloads it.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('cancellation_artifacts', 'zip', 'artifacts')
print('Contents:')
!ls -la artifacts
files.download('cancellation_artifacts.zip')

## Step 6 — Put the model back in your app
Unzip `cancellation_artifacts.zip` and copy its contents into **`backend/artifacts/`**, replacing what's there. The two that power predictions are:
- `cancellation_model.joblib`
- `feature_pipeline.joblib`

The JSONs (`model_metrics.json`, `threshold_recommendation.json`, `threshold_analysis.json`, `feature_importance.json`, `pr_curve.json`) drive the dashboard's model card, threshold view, and feature-importance chart.

Then restart the backend (`uvicorn app.main:app --reload --port 8000`). `GET /api/cancellation/model/info` should now report `available: true`, and the live queue will score with the model.